# Step 2 : 

Step1 では入力列$U$を与えて、状態軌道$X$と総コスト$J$を順方向に計算を行った。

Step2 では、その入力列$U$をどう決めれば総コスト$J$が最小になるかを、終端から逆向きに考える。

## LQR で扱う問題

まず、離散時間状態方程式を線形として考える。

$$
x_{k+1} = A_k x_k + B_k u_k
$$

コスト$J$は2次形式で次のようなものである。ここでは最初から離散時間で考えているため、ステージコスト$\ell_k$に$\Delta t$ は掛けていない。

$$
J = \phi(x_N) + \sum_{k=0}^{N-1} \ell_k ( x_k, u_k) = \frac{1}{2} {x_N}^T Q_N {x_N} + \sum_{k=0}^{N-1}\left( \frac{1}{2} {x_k}^T Q_k {x_k} + \frac{1}{2} {u_k}^T R_k {u_k} \right)
$$

ここでは簡単のため目標状態を$x_{ref}=0$ と原点としている。
- $Q_k$ : 状態を原点へ近づける重み (半正定値行列 $Q_k \succeq 0$)
- $R_k$ : 入力を小さくする重み (正定値行列 $R_k \succ 0$)
- $Q_N$ : 終端状態を原点へ近づける重み (半正定値行列 $Q_N \succeq 0$)

半正定値行列はその固有値が0以上であり、正定値行列はその固有値が正の値である行列である。例えば次のものである。

$$
Q_k = \begin{bmatrix}
1 & 0 \\
0 & 0
\end{bmatrix} , \quad
R_k = \begin{bmatrix}
1 & 0 \\
0 & 1
\end{bmatrix}
$$

これを制御的な意味で考えると、$Q_k \succeq 0$ によって評価しない状態あってもよい、であり、$R_k \succ 0$ によって入力は必ず評価するということなる。

$\ell_k$ と表示しているが、これは $R_k, Q_k$ のように時刻によりゲインを変更することが出来るためである。
例えば終端に近づくほど目標誤差を重視するように設定することが出来る。

ここではまだ 入力列 $U$ が決まっていない。

## 価値関数について

時刻 $k$ の状態 $x_k$ から終端までのコストを、残りの$U_k$ について最小化する。この時の最小コストが価値関数 $V_k(x_k)$ であり、それを実現する入力列が最適入力列 $U_k^*$ である。

$U_k$は以下のように$k$から$N-1$までの入力列である。

$$
U_k = \begin{bmatrix} u_k & u_{k+1} & \cdots & u_{N-1} \end{bmatrix}^T
$$

価値関数は$J_k$を$U_k$について最小化した後の最小コストである。($U_k$を変数として動かしたときに、最も小さい$J_k$の値)

$$
V_k(x_k) = \min\limits_{U_k} J_k(x_k, U_k) = \min\limits_{U_k} \left[ \sum_{i=k}^{N-1} \ell_i(x_i, u_i) + \phi(x_N) \right]
$$


一方、最小化すべき入力列は以下となる。(最小となる$J_k$の入力列$U_k$)

$$
U_k^* = \underset{U_k}{\arg \min} J_k (x_k, U_k)
$$


## Bellmanの最適性原理

時刻 $k$ から終端$N$までの最小コストは、次の2つの合計を、現在の入力 $u_k$ について最小化することで求められる。

1. 現在の入力 $u_k$ によって発生するステージコスト $\ell_k(x_k, u_k)$
1. $u_k$によって決まる次の状態 $x_{k+1}$ から先の最小コスト $V_{k+1}(x_{k+1})$

$$
V_k(x_k) = \min\limits_{u_k} \left[ \ell_k(x_k, u_k) + V_{k+1}(x_{k+1})\right]
$$

$V_{k+1}$は入力列 $u_{k+1} , \cdots, u_{N-1}$について最小化済みだが、その出発点$x_{k+1}$によって値が変化する。

$$
V_{k+1}(x_{k+1}) = \min\limits_{u_{k+1}, \cdots, u_{N-1}} J_{k+1}(x_{k+1}, u_{k+1}, \cdots, u_{N-1})
$$

$x_{k+1}$は以下であるため、$u_k$によって変化する。

$$
x_{k+1} = A_k x_k + B_k u_k
$$

よって、現在の入力 $u_k$ を選ぶときには、現在のコスト $\ell_k$ と遷移先からの最小コスト $V_{k+1}$ の合計が最小になる $u_k$ を選ぶ必要がある。

$$
V_k(x_k) = \min\limits_{u_k} \left[ \ell_k(x_k, u_k) + V_{k+1}(A_k x_k + B_k u_k) \right]
$$

この式をBellman方程式と呼ぶ。

重要なことは、未来の入力列全部を一度に考えるの代わりに、今の入力$u_k$と、次の時刻以降の最小コスト $V_{k+1}$に分けていることである。

## 終端から逆向きに価値関数を計算する理由

ステージコストは$k=0$から$k=N-1$までのため、終端時刻$N$では価値関数は$x_N$のみの関数になる。

$$
V_N(x_N) = \phi(x_N) =\frac{1}{2} {x_N}^T \ Q_N \ x_N
$$

$x_N$の具体的な値が分かっているわけではなく、終端価値関数 $V_N(x_N)$ の関数形が分かっているという意味である。逆向きに計算するbackward pass では状態を変数のまま扱う。

そして、$V_N(x_N)$が分かれば、一つ前の価値関数は次で計算できる。

$$
V_{N-1}(x_{N-1}) = \min\limits_{u_{N-1}} \left[ \ell_{N-1}(x_{N-1}, u_{N-1})  + V_N(x_N) \right]
$$

これに状態方程式を代入すると以下となり、$x_N$は$x_{N-1}$と$u_{N-1}$で求まることが分かる。

$$
V_{N-1}(x_{N-1}) = \min\limits_{u_{N-1}} \left[ \ell_{N-1}(x_{N-1}, u_{N-1})  + V_N(A_{N-1} x_{N-1} + B_{N-1} u_{N-1}) \right]
$$


さらに$V_{N-1}$が分かれば、$V_{N-2}$を計算でき、この処理を繰り返すことで価値関数を逆向きに計算できる。

$$
V_{N} \rightarrow V_{N-1} \rightarrow V_{N-2} \rightarrow \cdots \rightarrow V_{0}
$$

よって、逆向きに計算する理由は<br>

現在$k$から終端$N$までの最小コストを表す価値関数 $V_k(x_k)$ を求めるには、次の時刻$k+1$から終端$N$までの最小コスト $V_{k+1}(x_{k+1})$ が必要となる。その出発点となる終端価値関数 $V_N(x_N) = \phi(x_N)$ の関数形だけが最初から分かっているため、価値関数は終端から逆向きに計算する。

## フィードバックゲインとRiccati再帰の導出

価値関数は最適入力を計算することで以下の二次関数となる。これを導出する過程で、Riccati再帰を導出する。

$$
V_{k+1}(x_{k+1}) = \frac{1}{2} {x_{k+1}}^T \ P_{k+1} \ {x_{k+1}}
$$

### 価値関数が$U$の関数ではない理由

時刻 $k+1$ の状態 $x_{k+1}$ を初期状態として固定する。そして、入力列 $U_{k+1}$ を仮に以下のように与える。

$$
U_{k+1} = [u_{k+1}, u_{k+2}, \cdots , u_{N-1}]
$$

状態方程式によって以下のように終端状態 $x_N$ まで順方向に計算することができる。これをrollout と呼ぶ。

$$
\begin{aligned}
x_{k+2} &= A_{k+1} \ x_{k+1} + B_{k+1} \ u_{k+1} \\
x_{k+3} &= A_{k+2} \ x_{k+2} + B_{k+2} \ u_{k+2} \\
\end{aligned}
$$

そして、その入力列$U_{k+1}$は以下のコスト $J_{k+1}$ によって評価することが出来る。

$$
J_{k+1} = \sum_{i=k+1}^{N-1} \ell_i(x_i, u_i) + \phi(x_N)
$$

同じ$x_{k+1}$から開始し、入力列を${U_{k+1}}^{(i)}$のように変化させれば状態軌道$X$とコスト$J$が変化する。

$$
\begin{aligned}
{U_{k+1}}^{(1)} &\rightarrow X^{(1)} \rightarrow {J_{k+1}}^{(1)} \\
{U_{k+1}}^{(2)} &\rightarrow X^{(2)} \rightarrow {J_{k+1}}^{(2)} \\
& \vdots
\end{aligned}
$$

これらの中から、$J_{k+1}$を最小化する入力列を $U_{k+1}^*$ と定義する。

$$
U_{k+1}^* = \underset{U_{k+1}}{\arg \min} J_{k+1} (x_{k+1}, U_{k+1})
$$

そして、その時の最小コストが価値関数$V_{k+1}(x_{k+1})$である。

$$
V_{k+1}(x_{k+1}) = \underset{U_{k+1}}{\min} J_{k+1} (x_{k+1}, U_{k+1})
$$

例えば、原点に戻す制御を考えた場合、

- $x_{k+1}$が原点の右側なら、左向きの入力が必要
- $x_{k+1}$が原点の左側なら、右向きの入力が必要
- $x_{k+1}$が原点にあるなら、入力は不要

となるため、最小コストを実現するための最適入力列$U_{k+1}^*$は、開始状態$x_{k+1}$によって変化する。したがって、最適入力列を初期状態$x_{k+1}$の関数として捉えることが出来る。

$$
U_{k+1}^* = U_{k+1}^*(x_{k+1})
$$

この最適入力列をコスト $J_{k+1}$に戻すと、$J_{k+1}$は最小化され価値関数は次のようになる。

$$
V_{k+1}(x_{k+1}) = J_{k+1} (x_{k+1}, U_{k+1}^*)
$$

最適入力列$U_{k+1}^*$は$x_{k+1}$から決定されるため、結果的に価値関数は$x_{k+1}$だけの関数となる。

$$
x_{k+1} \longrightarrow U_{k+1}^*(x_{k+1}) \longrightarrow V_{k+1}(x_{k+1}) \text{(最小コスト)}
$$


### 最適入力列 $U_{k+1}^*$の計算

まず、終端$N$では入力がないため、以下となる。

$$
V_N(x_N) = \phi(x_N) = \frac{1}{2}x_N^T Q_N x_N
$$

よって、$P_N = Q_N$ と置くことが出来る。

終端のひとつ前の時刻 $N-1$では次となる。

$$
\begin{aligned}
V_{N-1}(x_{N-1}) &= \underset{u_{N-1}}{\min} J_{N-1} (x_{N-1}, u_{N-1}) \\
&= \min\limits_{u_{N-1}} \left[ \ell_{N-1}(x_{N-1}, u_{N-1})  + V_N(x_N) \right] \\
&= \underset{u_{N-1}}{\min} \left[ \frac{1}{2}x_{N-1}^T Q_{N-1} x_{N-1} + \frac{1}{2} u_{N-1}^T R_{N-1} u_{N-1} + \frac{1}{2}x_N^T P_N x_N \right]
\end{aligned}
$$

状態方程式より$x_N$は次のようになるため、これを代入する。

$$
x_{N} = A_{N-1} x_{N-1} + B_{N-1} u_{N-1}
$$

$V_{N-1}(x_{N-1})$の括弧内 $J_{N-1}$を示すと以下となり、$x_{N-1}$と$u_{N-1}$の二次形式になっている。

$$
\begin{aligned}
J_{N-1} = &\frac{1}{2}x_{N-1}^T Q_{N-1} x_{N-1} + \frac{1}{2} u_{N-1}^T R_{N-1} u_{N-1} \\
&+ \frac{1}{2} (A_{N-1} x_{N-1} + B_{N-1} u_{N-1})^T P_N (A_{N-1} x_{N-1} + B_{N-1} u_{N-1})
\end{aligned}
$$

$V_{N-1}(x_{N-1})$はその式が示すように、上式の二次形式を$u_{N-1}$について最小化する。

上式を$u_{N-1}$と$x_{N-1}$でまとめると以下となる。

$$
\begin{aligned}
J_{N-1} = &\frac{1}{2} u_{N-1}^T \ \left[ R_{N-1} + B_{N-1}^T P_N B_{N-1} \right] \ u_{N-1}  + u_{N+1}^T \left[ B_{N-1}^T P_N A_{N-1} \right] x_{N-1}\\
&+ \frac{1}{2} x_{N-1}^T \left[ Q_{N-1} + A_{N-1}^T P_N A_{N-1} \right] \ x_{N-1}
\end{aligned}
$$

$J_{N-1}$を$u_{N-1}$で2階変便を行うとHessian $H$ となる。

$$
\frac{\partial^2 J_{N-1}}{\partial u_{N-1}^2} = H = R_{N-1} + B_{N-1}^T P_N B_{N-1}
$$

この$H$が 

- 正定値 $H \succ 0$ であればどの方向にも曲がるおわん型の狭義凸関数となり、一意な最小値を持つ。
- 半正定値 $H \succeq 0$ でれば平らな方向が存在する可能があり、最小値が一意とは限らない
- 不定 でれば最小値とは限らない

$P_N = Q_N \succeq 0$ であり、$R_{N-1} \succ 0 $ とコスト関数では設定している。そのため、任意のゼロでないベクトル$z$に対し、2次形式を計算すると、 $z^T H z > 0$ となり$H$が正定値であることを示している。

$$
\begin{aligned}
z^T H z &= z^T  R_{N-1} z + z^T B_{N-1}^T P_N B_{N-1} z\\
&= \underset{ > 0 }{\underbrace{z^T  R_{N-1} z }} + \underset{ \ge 0 }{\underbrace{  (B_{N-1}z)^T P_N (B_{N-1} z)}}\\
& > 0
\end{aligned}
$$


よって、$J_{N-1}$ は$u_{N-1}$について狭義凸であり、偏微分がゼロになる点が一意な最小点となる。




$J_{N-1}$を$u_{N-1}$で偏微分してゼロとする。

$$
[R_{N-1} + B_{N-1}^T P_N B_{N-1}] u_{N-1} + B_{N-1} P_N A_{N-1} x_{N-1} = 0
$$

ここから$u_{N-1}$を求めると、それが$J_{N-1}$が最小となる最適入力$u_{N-1}^*$となる。$R_{N-1} + B_{N-1}^T P_N B_{N-1}$は正定値であるため逆行列が存在する。

$$
\begin{aligned}
u_{N-1} &= - \left\{ (R_{N-1} + B_{N-1}^T P_N B_{N-1} )^{-1} B_{N-1}^T P_N A_{N-1} \right\} x_{N-1}
\end{aligned}
$$

よって、最適入力$u_{N-1}^*$は以下となる。

$$
u_{N-1}^* = - K_{N-1} x_{N-1}, \quad K_{N-1} = (R_{N-1} + B_{N-1}^T P_N B_{N-1} )^{-1} B_{N-1}^T P_N A_{N-1}
$$

この$u_{N-1}^*$を $J_{N-1}$ に戻す。この時、$u_{N-1}^*$により$J_{N-1}$は最小化されているので次のよう$x_{N-1}$の関数になる。最適入力が$u_{N-1}^* = -K_{N-1} x_{N-1}$ と現在状態に比例する形で得られるため、$K_{N-1}$は状態フィードバックゲインである。

$$
\begin{aligned}
V_{N-1}(x_{N-1}) &= J_{N-1}(x_{N-1}, u_{N-1}^*) \\
&=  \frac{1}{2}x_{N-1}^T Q_{N-1} x_{N-1} + \frac{1}{2} (K_{N-1} x_{N-1})^T R_{N-1} (K_{N-1} x_{N-1}) \\
&+ \frac{1}{2} \left( ( A_{N-1} - B_{N-1} K_{N-1})  x_{N-1}) \right)^T P_N \left( ( A_{N-1} - B_{N-1} K_{N-1})  x_{N-1}) \right) \\
=&  \frac{1}{2}x_{N-1}^T \Big[ Q_{N-1} + K_{N-1}^T R_{N-1} K_{N-1} \\
& \qquad + \left(  A_{N-1} - B_{N-1} K_{N-1} \right)^T P_N \left(  A_{N-1} - B_{N-1} K_{N-1} \right) \Big]  x_{N-1}
\end{aligned}
$$

$[ \ ]$括弧の中を$P_{N-1}$とおく。

$$
\begin{aligned}
P_{N-1} &= Q_{N-1} + K_{N-1}^T R_{N-1} K_{N-1} \\
& \qquad + \left( A_{N-1} - B_{N-1} K_{N-1} \right)^T P_N \left( A_{N-1} - B_{N-1} K_{N-1} \right) 
\end{aligned}
$$

この$P_{N-1}$を用いて、$V_{N-1}$は次のようになる。

$$
V_{N-1}(x_{N-1}) = \frac{1}{2} x_{N-1}^T P_{N-1} x_{N-1}
$$

よって、$V_{N-1}$は、ひとつ前の $P_N$ から $K_{N-1}$ を計算し、$K_{N-1}$を用いた最適入力 $u_{N-1}^*$ により $P_{N-1}$ を構成して$V_{N-1}$を求めた。

$$
P_N \rightarrow K_{N-1} \rightarrow u_{N-1}^* \rightarrow P_{N-1} \rightarrow V_{N-1}
$$


同じく、$N-2$、$N-3$とそれぞれにおいて$K_{N-2}, K_{N-3}$と最適入力 $u_{N-2}^*, u_{N-3}^*$ と順に計算していくことで以下の価値関数を終端$N$から逆方向に計算することが出来る。

$$
\begin{aligned}
V_{N-2}(x_{N-2}) &= \frac{1}{2} x_{N-2}^T P_{N-2} x_{N-2} \\
V_{N-3}(x_{N-3}) &= \frac{1}{2} x_{N-3}^T P_{N-3} x_{N-3} \\
& \ \vdots
\end{aligned}
$$

これは以下のような流れである。

$$
\begin{aligned}
P_N & \rightarrow K_{N-1} \rightarrow u_{N-1}^* \rightarrow P_{N-1} \\
& \rightarrow K_{N-2} \rightarrow u_{N-2}^* \rightarrow P_{N-2} \\
& \rightarrow K_{N-3} \rightarrow u_{N-3}^* \rightarrow P_{N-3} \rightarrow \cdots 
\end{aligned}
$$


$k$時点の計算式を以下に示す。

$$
\begin{aligned}
K_{k} &= (R_{k} + B_{k}^T P_{k+1} B_{k} )^{-1} B_{k}^T P_{k+1} A_{k} \\
P_{k} &= Q_{k} + K_{k}^T R_{k} K_{k} \\
& \qquad + \left( A_{k} - B_{k} K_{k} \right)^T P_{k+1} \left(  A_{k} - B_{k} K_{k}  \right) 
\end{aligned}
$$

$K_{k}$ はフィードバックゲインであり、$P_{k+1}$から$P_{k}$を終端から逆向きに計算する処理をRiccati再帰と呼ぶ。

上記のように、最適入力列$U_{k+1}^*$を求めることは、フィードバックゲインとRiccati再帰を計算することであるといえる。

上記の処理をbackword pass と呼ぶ。

ここで重要なことは、具体的な $x_{k}$ がまだ決定していないということである。上記で求めたのは計算の方法である。
